# Import Libraries

In [5]:
import pandas as pd  # for datahandling 
import numpy as np   # for numerical operations

from sklearn.model_selection import train_test_split   # for splitting the data
from sklearn.preprocessing import StandardScaler       #for scaling numerical features

# Load Datasets

In [9]:
matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

print("Matches shape:", matches.shape)
print("Deliveries shape:", deliveries.shape)

matches.head()

Matches shape: (636, 18)
Deliveries shape: (150460, 21)


,id,season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,2017,Hyderabad,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,2,2017,Pune,2017-04-06,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,normal,0,Rising Pune Supergiant,0,7,SPD Smith,Maharashtra Cricket Association Stadium,A Nand Kishore,S Ravi,NaN
2,3,2017,Rajkot,2017-04-07,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,normal,0,Kolkata Knight Riders,0,10,CA Lynn,Saurashtra Cricket Association Stadium,Nitin Menon,CK Nandan,NaN
3,4,2017,Indore,2017-04-08,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab,field,normal,0,Kings XI Punjab,0,6,GJ Maxwell,Holkar Cricket Stadium,AK Chaudhary,C Shamshuddin,NaN
4,5,2017,Bangalore,2017-04-08,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,normal,0,Royal Challengers Bangalore,15,0,KM Jadhav,M Chinnaswamy Stadium,NaN,NaN,NaN


In [10]:
deliveries.head()

,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,bye_runs,legbye_runs,noball_runs,penalty_runs,batsman_runs,extra_runs,total_runs,player_dismissed,dismissal_kind,fielder
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,4,0,4,NaN,NaN,NaN
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,0,0,NaN,NaN,NaN
4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,S Dhawan,TS Mills,0,...,0,0,0,0,0,2,2,NaN,NaN,NaN


# Inspect the data

In [11]:
print("Matches columns:\n", matches.columns.tolist())
print("\nDeliveries columns:\n", deliveries.columns.tolist())

Matches columns:
 ['id', 'season', 'city', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs', 'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3']

Deliveries columns:
 ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs', 'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs', 'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed', 'dismissal_kind', 'fielder']


# Clean the Matches Dataset

In [12]:
# Drop unnecessary columns
matches = matches.drop(columns=["umpire1", "umpire2", "umpire3", "player_of_match"], errors="ignore")

# Remove no-result matches because the target outcome is unclear
matches = matches[matches["result"] != "no result"].copy()

# Standardize inconsistent team names
matches.replace("Rising Pune Supergiants", "Rising Pune Supergiant", inplace=True)
deliveries.replace("Rising Pune Supergiants", "Rising Pune Supergiant", inplace=True)

# Fill missing city values
matches["city"] = matches["city"].fillna("Dubai")

print("Cleaned matches shape:", matches.shape)
matches.head()

Cleaned matches shape: (633, 14)


,id,season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,venue
0,1,2017,Hyderabad,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Sunrisers Hyderabad,35,0,"Rajiv Gandhi International Stadium, Uppal"
1,2,2017,Pune,2017-04-06,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,normal,0,Rising Pune Supergiant,0,7,Maharashtra Cricket Association Stadium
2,3,2017,Rajkot,2017-04-07,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,normal,0,Kolkata Knight Riders,0,10,Saurashtra Cricket Association Stadium
3,4,2017,Indore,2017-04-08,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab,field,normal,0,Kings XI Punjab,0,6,Holkar Cricket Stadium
4,5,2017,Bangalore,2017-04-08,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,normal,0,Royal Challengers Bangalore,15,0,M Chinnaswamy Stadium


# Convert Date and Create Match Features

In [13]:
# Convert date column to datetime
matches["date"] = pd.to_datetime(matches["date"])

# Extract useful date-related features
matches["match_month"] = matches["date"].dt.month
matches["match_dayofweek"] = matches["date"].dt.dayofweek

# Keep original match date for optional chronological split later
matches["match_date"] = matches["date"]

# Toss-related features
matches["toss_decision_bat"] = (matches["toss_decision"] == "bat").astype(int)
matches["toss_winner_is_team1"] = (matches["toss_winner"] == matches["team1"]).astype(int)

# Target variable: did team1 win?
matches["team1_won"] = (matches["winner"] == matches["team1"]).astype(int)

matches.head()

,id,season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,venue,match_month,match_dayofweek,match_date,toss_decision_bat,toss_winner_is_team1,team1_won
0,1,2017,Hyderabad,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Sunrisers Hyderabad,35,0,"Rajiv Gandhi International Stadium, Uppal",4,2,2017-04-05,0,0,1
1,2,2017,Pune,2017-04-06,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,normal,0,Rising Pune Supergiant,0,7,Maharashtra Cricket Association Stadium,4,3,2017-04-06,0,0,0
2,3,2017,Rajkot,2017-04-07,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,normal,0,Kolkata Knight Riders,0,10,Saurashtra Cricket Association Stadium,4,4,2017-04-07,0,0,0
3,4,2017,Indore,2017-04-08,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab,field,normal,0,Kings XI Punjab,0,6,Holkar Cricket Stadium,4,5,2017-04-08,0,0,0
4,5,2017,Bangalore,2017-04-08,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,normal,0,Royal Challengers Bangalore,15,0,M Chinnaswamy Stadium,4,5,2017-04-08,1,1,1


# Build Over-Level Features

In [16]:
over_stats = deliveries.groupby(
    ["match_id", "inning", "over", "batting_team", "bowling_team"],
    as_index=False
).agg(
    runs_in_over=("total_runs", "sum"),
    balls_in_over=("ball", "count"),
    wickets_in_over=("player_dismissed", lambda x: x.notna().sum()),
    extras_in_over=("extra_runs", "sum"),
    fours_in_over=("batsman_runs", lambda x: (x == 4).sum()),
    sixes_in_over=("batsman_runs", lambda x: (x == 6).sum()),
    dots_in_over=("total_runs", lambda x: (x == 0).sum())
)

print("Over-level dataset shape:", over_stats.shape)
over_stats.head()

Over-level dataset shape: (24388, 12)


,match_id,inning,over,batting_team,bowling_team,runs_in_over,balls_in_over,wickets_in_over,extras_in_over,fours_in_over,sixes_in_over,dots_in_over
0,1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,7,7,0,3,1,0,4
1,1,1,2,Sunrisers Hyderabad,Royal Challengers Bangalore,16,7,1,1,2,1,2
2,1,1,3,Sunrisers Hyderabad,Royal Challengers Bangalore,6,6,0,0,0,0,2
3,1,1,4,Sunrisers Hyderabad,Royal Challengers Bangalore,4,6,0,0,0,0,2
4,1,1,5,Sunrisers Hyderabad,Royal Challengers Bangalore,9,6,0,0,1,0,1


# Create Cumulative Features

In [17]:
# Sort before cumulative calculations
over_stats = over_stats.sort_values(["match_id", "inning", "over"]).reset_index(drop=True)

# Cumulative features
over_stats["cum_runs"] = over_stats.groupby(["match_id", "inning"])["runs_in_over"].cumsum()
over_stats["cum_wickets"] = over_stats.groupby(["match_id", "inning"])["wickets_in_over"].cumsum()
over_stats["cum_extras"] = over_stats.groupby(["match_id", "inning"])["extras_in_over"].cumsum()
over_stats["cum_fours"] = over_stats.groupby(["match_id", "inning"])["fours_in_over"].cumsum()
over_stats["cum_sixes"] = over_stats.groupby(["match_id", "inning"])["sixes_in_over"].cumsum()
over_stats["cum_dots"] = over_stats.groupby(["match_id", "inning"])["dots_in_over"].cumsum()

# Current run rate
over_stats["run_rate"] = over_stats["cum_runs"] / over_stats["over"]

# Wickets remaining
over_stats["wickets_remaining"] = 10 - over_stats["cum_wickets"]

over_stats.head()


,match_id,inning,over,batting_team,bowling_team,runs_in_over,balls_in_over,wickets_in_over,extras_in_over,fours_in_over,sixes_in_over,dots_in_over,cum_runs,cum_wickets,cum_extras,cum_fours,cum_sixes,cum_dots,run_rate,wickets_remaining
0,1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,7,7,0,3,1,0,4,7,0,3,1,0,4,7.000000,10
1,1,1,2,Sunrisers Hyderabad,Royal Challengers Bangalore,16,7,1,1,2,1,2,23,1,4,3,1,6,11.500000,9
2,1,1,3,Sunrisers Hyderabad,Royal Challengers Bangalore,6,6,0,0,0,0,2,29,1,4,3,1,8,9.666667,9
3,1,1,4,Sunrisers Hyderabad,Royal Challengers Bangalore,4,6,0,0,0,0,2,33,1,4,3,1,10,8.250000,9
4,1,1,5,Sunrisers Hyderabad,Royal Challengers Bangalore,9,6,0,0,1,0,1,42,1,4,4,1,11,8.400000,9


# 

# Rename Key Column for Merge

In [18]:
over_stats = over_stats.rename(columns={"match_id": "id"})
over_stats.head()

,id,inning,over,batting_team,bowling_team,runs_in_over,balls_in_over,wickets_in_over,extras_in_over,fours_in_over,sixes_in_over,dots_in_over,cum_runs,cum_wickets,cum_extras,cum_fours,cum_sixes,cum_dots,run_rate,wickets_remaining
0,1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,7,7,0,3,1,0,4,7,0,3,1,0,4,7.000000,10
1,1,1,2,Sunrisers Hyderabad,Royal Challengers Bangalore,16,7,1,1,2,1,2,23,1,4,3,1,6,11.500000,9
2,1,1,3,Sunrisers Hyderabad,Royal Challengers Bangalore,6,6,0,0,0,0,2,29,1,4,3,1,8,9.666667,9
3,1,1,4,Sunrisers Hyderabad,Royal Challengers Bangalore,4,6,0,0,0,0,2,33,1,4,3,1,10,8.250000,9
4,1,1,5,Sunrisers Hyderabad,Royal Challengers Bangalore,9,6,0,0,1,0,1,42,1,4,4,1,11,8.400000,9


# Select Safe Match Metadata

In [19]:
# IMPORTANT:
# Do NOT include columns like:
# winner, result, win_by_runs, win_by_wickets
# because they reveal the final outcome and cause data leakage.

match_meta = matches[
    [
        "id",
        "season",
        "city",
        "team1",
        "team2",
        "venue",
        "toss_winner",
        "toss_decision",
        "toss_winner_is_team1",
        "toss_decision_bat",
        "dl_applied",
        "match_month",
        "match_dayofweek",
        "match_date",
        "team1_won"
    ]
].copy()

match_meta.head()

,id,season,city,team1,team2,venue,toss_winner,toss_decision,toss_winner_is_team1,toss_decision_bat,dl_applied,match_month,match_dayofweek,match_date,team1_won
0,1,2017,Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,field,0,0,0,4,2,2017-04-05,1
1,2,2017,Pune,Mumbai Indians,Rising Pune Supergiant,Maharashtra Cricket Association Stadium,Rising Pune Supergiant,field,0,0,0,4,3,2017-04-06,0
2,3,2017,Rajkot,Gujarat Lions,Kolkata Knight Riders,Saurashtra Cricket Association Stadium,Kolkata Knight Riders,field,0,0,0,4,4,2017-04-07,0
3,4,2017,Indore,Rising Pune Supergiant,Kings XI Punjab,Holkar Cricket Stadium,Kings XI Punjab,field,0,0,0,4,5,2017-04-08,0
4,5,2017,Bangalore,Royal Challengers Bangalore,Delhi Daredevils,M Chinnaswamy Stadium,Royal Challengers Bangalore,bat,1,1,0,4,5,2017-04-08,1


# Merge Over-Level Data with Match Data

In [20]:
df = over_stats.merge(match_meta, on="id", how="inner")

# Keep only innings 1 and 2
df = df[df["inning"].isin([1, 2])].copy()

print("Merged dataset shape:", df.shape)
df.head()

Merged dataset shape: (24321, 34)


,id,inning,over,batting_team,bowling_team,runs_in_over,balls_in_over,wickets_in_over,extras_in_over,fours_in_over,...,venue,toss_winner,toss_decision,toss_winner_is_team1,toss_decision_bat,dl_applied,match_month,match_dayofweek,match_date,team1_won
0,1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,7,7,0,3,1,...,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,field,0,0,0,4,2,2017-04-05,1
1,1,1,2,Sunrisers Hyderabad,Royal Challengers Bangalore,16,7,1,1,2,...,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,field,0,0,0,4,2,2017-04-05,1
2,1,1,3,Sunrisers Hyderabad,Royal Challengers Bangalore,6,6,0,0,0,...,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,field,0,0,0,4,2,2017-04-05,1
3,1,1,4,Sunrisers Hyderabad,Royal Challengers Bangalore,4,6,0,0,0,...,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,field,0,0,0,4,2,2017-04-05,1
4,1,1,5,Sunrisers Hyderabad,Royal Challengers Bangalore,9,6,0,0,1,...,"Rajiv Gandhi International Stadium, Uppal",Royal Challengers Bangalore,field,0,0,0,4,2,2017-04-05,1


# Check Null Values and Structure

In [21]:
print("Null values:\n")
print(df.isnull().sum())

print("\nNumber of unique matches:", df["id"].nunique())
print("Target distribution:\n")
print(df["team1_won"].value_counts())

Null values:

id                      0
inning                  0
over                    0
batting_team            0
bowling_team            0
runs_in_over            0
balls_in_over           0
wickets_in_over         0
extras_in_over          0
fours_in_over           0
sixes_in_over           0
dots_in_over            0
cum_runs                0
cum_wickets             0
cum_extras              0
cum_fours               0
cum_sixes               0
cum_dots                0
run_rate                0
wickets_remaining       0
season                  0
city                    0
team1                   0
team2                   0
venue                   0
toss_winner             0
toss_decision           0
toss_winner_is_team1    0
toss_decision_bat       0
dl_applied              0
match_month             0
match_dayofweek         0
match_date              0
team1_won               0
dtype: int64

Number of unique matches: 633
Target distribution:

team1_won
0    13147
1    11174
Name

# Optional Restriction to Early Overs

In [22]:
# If you want a more realistic prediction problem,
# you may restrict the dataset to earlier overs only.
# This avoids making the task too easy using near-final game states.

# Example:
# df = df[df["over"] <= 15].copy()

print("Current dataset shape:", df.shape)

Current dataset shape: (24321, 34)


# Split by Match ID

In [23]:
# This is critical to avoid leakage.
# All overs from the same match must stay in either train or test.

match_level_target = df.groupby("id")["team1_won"].first()

train_ids, test_ids = train_test_split(
    match_level_target.index,
    test_size=0.2,
    random_state=42,
    stratify=match_level_target.values
)

train_df = df[df["id"].isin(train_ids)].copy()
test_df = df[df["id"].isin(test_ids)].copy()

print("Train rows:", train_df.shape)
print("Test rows:", test_df.shape)
print("Train matches:", train_df["id"].nunique())
print("Test matches:", test_df["id"].nunique())

Train rows: (19431, 34)
Test rows: (4890, 34)
Train matches: 506
Test matches: 127


# Check Train/Test Overlap

In [24]:
overlap = set(train_ids).intersection(set(test_ids))
print("Overlap between train and test match IDs:", overlap)

Overlap between train and test match IDs: set()


# Encode Categorical Columns

In [25]:
cat_cols = [
    "city",
    "team1",
    "team2",
    "batting_team",
    "bowling_team",
    "toss_winner",
    "toss_decision",
    "venue"
]

category_maps = {}

for col in cat_cols:
    unique_train_values = sorted(train_df[col].astype(str).unique())
    mapping = {value: idx for idx, value in enumerate(unique_train_values)}
    category_maps[col] = mapping

    train_df[col + "_enc"] = train_df[col].astype(str).map(mapping).fillna(-1).astype(int)
    test_df[col + "_enc"] = test_df[col].astype(str).map(mapping).fillna(-1).astype(int)

print("Encoded categorical columns successfully.")
train_df[[col + "_enc" for col in cat_cols]].head()

Encoded categorical columns successfully.


,city_enc,team1_enc,team2_enc,batting_team_enc,bowling_team_enc,toss_winner_enc,toss_decision_enc,venue_enc
0,14,12,11,12,11,11,1,23
1,14,12,11,12,11,11,1,23
2,14,12,11,12,11,11,1,23
3,14,12,11,12,11,11,1,23
4,14,12,11,12,11,11,1,23


# Scale Numeric Features

In [26]:
numeric_cols = [
    "over",
    "inning",
    "runs_in_over",
    "balls_in_over",
    "wickets_in_over",
    "extras_in_over",
    "fours_in_over",
    "sixes_in_over",
    "dots_in_over",
    "cum_runs",
    "cum_wickets",
    "cum_extras",
    "cum_fours",
    "cum_sixes",
    "cum_dots",
    "run_rate",
    "wickets_remaining",
    "season",
    "match_month",
    "match_dayofweek",
    "dl_applied"
]

scaler = StandardScaler()

train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols].fillna(0))
test_df[numeric_cols] = scaler.transform(test_df[numeric_cols].fillna(0))

print("Scaled numeric columns successfully.")
train_df[numeric_cols].head()

Scaled numeric columns successfully.


,over,inning,runs_in_over,balls_in_over,wickets_in_over,extras_in_over,fours_in_over,sixes_in_over,dots_in_over,cum_runs,...,cum_extras,cum_fours,cum_sixes,cum_dots,run_rate,wickets_remaining,season,match_month,match_dayofweek,dl_applied
0,-1.623114,-0.969934,-0.220978,1.313743,-0.564145,2.783444,0.364336,-0.485620,1.321782,-1.459448,...,-0.373575,-1.409405,-0.900600,-1.606646,-0.185486,1.194365,1.601403,-0.73679,-0.752854,-0.12981
1,-1.447029,-0.969934,1.765372,1.313743,1.292231,0.619347,1.580105,1.300164,-0.130784,-1.126935,...,-0.116658,-0.980263,-0.487368,-1.453722,2.052855,0.730628,1.601403,-0.73679,-0.752854,-0.12981
2,-1.270944,-0.969934,-0.441684,-0.269937,-0.564145,-0.462701,-0.851432,-0.485620,-0.130784,-1.002243,...,-0.116658,-0.980263,-0.487368,-1.300798,1.140938,0.730628,1.601403,-0.73679,-0.752854,-0.12981
3,-1.094859,-0.969934,-0.883095,-0.269937,-0.564145,-0.462701,-0.851432,-0.485620,-0.130784,-0.919115,...,-0.116658,-0.980263,-0.487368,-1.147874,0.436275,0.730628,1.601403,-0.73679,-0.752854,-0.12981
4,-0.918774,-0.969934,0.220433,-0.269937,-0.564145,-0.462701,0.364336,-0.485620,-0.857067,-0.732076,...,-0.116658,-0.765692,-0.487368,-1.071412,0.510887,0.730628,1.601403,-0.73679,-0.752854,-0.12981


# Build Final Train and Test Tables

In [27]:
final_cols = [
    "id",
    "inning",
    "over",
    "team1_won",
    "toss_winner_is_team1",
    "toss_decision_bat"
] + numeric_cols + [col + "_enc" for col in cat_cols]

final_train_df = train_df[final_cols].copy()
final_test_df = test_df[final_cols].copy()

print("Final train shape:", final_train_df.shape)
print("Final test shape:", final_test_df.shape)

final_train_df.head()

Final train shape: (19431, 35)
Final test shape: (4890, 35)


,id,inning,over,team1_won,toss_winner_is_team1,toss_decision_bat,over,inning,runs_in_over,balls_in_over,...,match_dayofweek,dl_applied,city_enc,team1_enc,team2_enc,batting_team_enc,bowling_team_enc,toss_winner_enc,toss_decision_enc,venue_enc
0,1,-0.969934,-1.623114,1,0,0,-1.623114,-0.969934,-0.220978,1.313743,...,-0.752854,-0.12981,14,12,11,12,11,11,1,23
1,1,-0.969934,-1.447029,1,0,0,-1.447029,-0.969934,1.765372,1.313743,...,-0.752854,-0.12981,14,12,11,12,11,11,1,23
2,1,-0.969934,-1.270944,1,0,0,-1.270944,-0.969934,-0.441684,-0.269937,...,-0.752854,-0.12981,14,12,11,12,11,11,1,23
3,1,-0.969934,-1.094859,1,0,0,-1.094859,-0.969934,-0.883095,-0.269937,...,-0.752854,-0.12981,14,12,11,12,11,11,1,23
4,1,-0.969934,-0.918774,1,0,0,-0.918774,-0.969934,0.220433,-0.269937,...,-0.752854,-0.12981,14,12,11,12,11,11,1,23


# Save Final Files

In [28]:
final_train_df.to_csv("ipl_train_over_level.csv", index=False)
final_test_df.to_csv("ipl_test_over_level.csv", index=False)

print("Files saved successfully:")
print("- ipl_train_over_level.csv")
print("- ipl_test_over_level.csv")

Files saved successfully:
- ipl_train_over_level.csv
- ipl_test_over_level.csv


# Final Summary

In [29]:
print("Final Train Shape:", final_train_df.shape)
print("Final Test Shape:", final_test_df.shape)

print("\nTrain target distribution:")
print(final_train_df["team1_won"].value_counts())

print("\nTest target distribution:")
print(final_test_df["team1_won"].value_counts())

Final Train Shape: (19431, 35)
Final Test Shape: (4890, 35)

Train target distribution:
team1_won
0    10478
1     8953
Name: count, dtype: int64

Test target distribution:
team1_won
0    2669
1    2221
Name: count, dtype: int64


# Key preprocessing decisions

## The dataset was transformed from match-level to over-level
One row represents the game state at the end of a given over,
Data was split by match ID to prevent leakage,
Outcome-revealing columns were excluded,
Numeric features were scaled using training data only,
Categorical features were encoded using train-only mappings

In [1]:
"""
=============================================================
  IPL Preprocessing — Over-Level Dataset
  Input  : matches.csv + deliveries.csv
  Output : ipl_preprocessed.csv
  Rows   : ~24,388  (vs 633 match-level — 38x more data)
=============================================================

WHY OVER-LEVEL?
  Match-level gives only 633 rows — too small for SVM/RF.
  By aggregating deliveries per OVER, every match contributes
  ~38 rows (20 overs x 2 innings). Same target label (who won
  the match), but 38x more training samples.

  Each row answers: "given the game state at the end of over N,
  who eventually won this match?"
=============================================================
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("=" * 60)
print("  IPL PREPROCESSING — OVER-LEVEL DATASET")
print("=" * 60)

# ─────────────────────────────────────────────────────────────
# STEP 1: LOAD BOTH FILES
# ─────────────────────────────────────────────────────────────
print("\n[STEP 1] Loading files ...")

matches    = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

print(f"  matches.csv    -> {matches.shape[0]:,} rows x {matches.shape[1]} columns")
print(f"  deliveries.csv -> {deliveries.shape[0]:,} rows x {deliveries.shape[1]} columns")


# ─────────────────────────────────────────────────────────────
# STEP 2: CLEAN matches.csv
# ─────────────────────────────────────────────────────────────
print("\n[STEP 2] Cleaning matches.csv ...")

# Drop useless/leaky columns
matches.drop(columns=["umpire1", "umpire2", "umpire3", "player_of_match"], inplace=True)

# Remove abandoned matches (no winner)
matches = matches[matches["result"] != "no result"].copy()

# Fix team name typo
matches.replace("Rising Pune Supergiants", "Rising Pune Supergiant", inplace=True)
deliveries.replace("Rising Pune Supergiants", "Rising Pune Supergiant", inplace=True)

# Fill 7 missing cities (all Dubai International Cricket Stadium)
matches["city"] = matches["city"].fillna("Dubai")

# Extract month and weekday from date, then drop raw date
matches["date"]            = pd.to_datetime(matches["date"])
matches["match_month"]     = matches["date"].dt.month
matches["match_dayofweek"] = matches["date"].dt.dayofweek
matches.drop(columns=["date"], inplace=True)

# Encode toss info as binary flags
matches["toss_decision_bat"]    = (matches["toss_decision"] == "bat").astype(int)
matches["toss_winner_is_team1"] = (matches["toss_winner"] == matches["team1"]).astype(int)

# TARGET: did team1 win this match?
matches["team1_won"] = (matches["winner"] == matches["team1"]).astype(int)

print(f"  Cleaned matches: {len(matches):,} rows")


# ─────────────────────────────────────────────────────────────
# STEP 3: BUILD OVER-LEVEL FEATURES FROM deliveries.csv
# ─────────────────────────────────────────────────────────────
print("\n[STEP 3] Aggregating deliveries -> per over stats ...")

over_groups = deliveries.groupby(
    ["match_id", "inning", "over", "batting_team", "bowling_team"]
)

over_stats = over_groups.agg(
    runs_in_over    = ("total_runs",       "sum"),
    balls_in_over   = ("ball",             "count"),
    wickets_in_over = ("player_dismissed", lambda x: x.notna().sum()),
    extras_in_over  = ("extra_runs",       "sum"),
    fours_in_over   = ("batsman_runs",     lambda x: (x == 4).sum()),
    sixes_in_over   = ("batsman_runs",     lambda x: (x == 6).sum()),
    dots_in_over    = ("total_runs",       lambda x: (x == 0).sum()),
).reset_index()

print(f"  Over-level rows: {len(over_stats):,}")

# Sort so cumulative sums run correctly
over_stats = over_stats.sort_values(
    ["match_id", "inning", "over"]
).reset_index(drop=True)

# Cumulative stats up to and including each over
over_stats["cum_runs"]    = over_stats.groupby(["match_id", "inning"])["runs_in_over"].cumsum()
over_stats["cum_wickets"] = over_stats.groupby(["match_id", "inning"])["wickets_in_over"].cumsum()
over_stats["cum_extras"]  = over_stats.groupby(["match_id", "inning"])["extras_in_over"].cumsum()
over_stats["cum_fours"]   = over_stats.groupby(["match_id", "inning"])["fours_in_over"].cumsum()
over_stats["cum_sixes"]   = over_stats.groupby(["match_id", "inning"])["sixes_in_over"].cumsum()
over_stats["cum_dots"]    = over_stats.groupby(["match_id", "inning"])["dots_in_over"].cumsum()

# Current run rate = runs scored so far / overs completed
over_stats["run_rate"] = (
    over_stats["cum_runs"] / over_stats["over"]
).round(4)

# Wickets remaining
over_stats["wickets_remaining"] = 10 - over_stats["cum_wickets"]

print("  Computed cumulative features: cum_runs, cum_wickets, run_rate, wickets_remaining ...")

# Rename for merge
over_stats.rename(columns={"match_id": "id"}, inplace=True)


# ─────────────────────────────────────────────────────────────
# STEP 4: MERGE over stats with match metadata
# ─────────────────────────────────────────────────────────────
print("\n[STEP 4] Merging over-level data with match metadata ...")

match_meta = matches[[
    "id", "season", "city", "team1", "team2", "venue",
    "toss_winner", "toss_decision", "toss_winner_is_team1",
    "toss_decision_bat", "dl_applied",
    "match_month", "match_dayofweek",
    "win_by_runs", "win_by_wickets",
    "team1_won"
]]

df = over_stats.merge(match_meta, on="id", how="inner")

# Keep only normal innings (1 and 2), drop super overs
df = df[df["inning"].isin([1, 2])].copy()

print(f"  Final merged shape: {df.shape[0]:,} rows x {df.shape[1]} columns")


# ─────────────────────────────────────────────────────────────
# STEP 5: ENCODE CATEGORICALS
# ─────────────────────────────────────────────────────────────
print("\n[STEP 5] Encoding categorical columns ...")

cat_cols = [
    "city", "team1", "team2", "batting_team",
    "bowling_team", "toss_winner", "toss_decision", "venue"
]

le = LabelEncoder()
for col in cat_cols:
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))
    print(f"  Encoded: {col:<20} -> {df[col].nunique()} unique values")


# ─────────────────────────────────────────────────────────────
# STEP 6: SCALE NUMERIC FEATURES
# ─────────────────────────────────────────────────────────────
print("\n[STEP 6] Scaling numeric features ...")

numeric_cols = [
    "over", "inning",
    "runs_in_over", "balls_in_over", "wickets_in_over",
    "extras_in_over", "fours_in_over", "sixes_in_over", "dots_in_over",
    "cum_runs", "cum_wickets", "cum_extras",
    "cum_fours", "cum_sixes", "cum_dots",
    "run_rate", "wickets_remaining",
    "season", "match_month", "match_dayofweek",
    "dl_applied", "win_by_runs", "win_by_wickets"
]

scaler = StandardScaler()
scaled_vals = scaler.fit_transform(df[numeric_cols].fillna(0))
scaled_df   = pd.DataFrame(
    scaled_vals,
    columns=[c + "_scaled" for c in numeric_cols],
    index=df.index
)
print(f"  Scaled {len(numeric_cols)} numeric columns -> *_scaled")


# ─────────────────────────────────────────────────────────────
# STEP 7: BUILD FINAL DATASET
# ─────────────────────────────────────────────────────────────
print("\n[STEP 7] Assembling final dataset ...")

id_cols     = ["id", "inning", "over"]
binary_cols = ["toss_winner_is_team1", "toss_decision_bat", "team1_won"]
enc_cols    = [c + "_enc" for c in cat_cols]
scl_cols    = [c + "_scaled" for c in numeric_cols]

final_df = pd.concat([
    df[id_cols].reset_index(drop=True),
    df[binary_cols].reset_index(drop=True),
    df[enc_cols].reset_index(drop=True),
    scaled_df.reset_index(drop=True)
], axis=1)


# ─────────────────────────────────────────────────────────────
# STEP 8: VALIDATE + SAVE
# ─────────────────────────────────────────────────────────────
print("\n[STEP 8] Validating and saving ...")
print(f"  Final shape      : {final_df.shape[0]:,} rows x {final_df.shape[1]} columns")
print(f"  Total null cells : {final_df.isnull().sum().sum()}")
print(f"  Target balance   : team1_won=1 -> {final_df['team1_won'].sum():,}  |  "
      f"team1_won=0 -> {(final_df['team1_won']==0).sum():,}")

print(f"\n  All {final_df.shape[1]} columns:")
for col in final_df.columns:
    print(f"    * {col}")

output_path = "ipl_preprocessed(final).csv"
final_df.to_csv(output_path, index=False)

print(f"\n{'='*60}")
print(f"  Saved -> {output_path}")
print(f"  Rows : {final_df.shape[0]:,}  (was 633 at match-level)")
print(f"  Cols : {final_df.shape[1]}")
print(f"  Train set (80%) : ~{int(final_df.shape[0]*0.8):,} rows")
print(f"  Test  set (20%) : ~{int(final_df.shape[0]*0.2):,} rows")
print(f"{'='*60}")

  IPL PREPROCESSING — OVER-LEVEL DATASET

[STEP 1] Loading files ...
  matches.csv    -> 636 rows x 18 columns
  deliveries.csv -> 150,460 rows x 21 columns

[STEP 2] Cleaning matches.csv ...
  Cleaned matches: 633 rows

[STEP 3] Aggregating deliveries -> per over stats ...
  Over-level rows: 24,388
  Computed cumulative features: cum_runs, cum_wickets, run_rate, wickets_remaining ...

[STEP 4] Merging over-level data with match metadata ...
  Final merged shape: 24,321 rows x 35 columns

[STEP 5] Encoding categorical columns ...
  Encoded: city                 -> 31 unique values
  Encoded: team1                -> 13 unique values
  Encoded: team2                -> 13 unique values
  Encoded: batting_team         -> 13 unique values
  Encoded: bowling_team         -> 13 unique values
  Encoded: toss_winner          -> 13 unique values
  Encoded: toss_decision        -> 2 unique values
  Encoded: venue                -> 35 unique values

[STEP 6] Scaling numeric features ...
  Scaled 2